# Fundamentals Alpha — Which Fundamentals Predict Future Returns?

**Goal:** Identify which fundamental characteristics (valuation, quality, profitability) predict
forward stock returns over 6-month, 1-year, 2-year, 3-year, and 5-year horizons.

**Data:** `monthly_pe` table — 296K rows, 1,414 tickers, 1999–2026 (monthly snapshots of
fundamental ratios and price).

**Approach** (mirrors `ibd50_analysis.ipynb`):
1. Compute forward returns from monthly price data
2. EDA — correlation heatmap, quintile return analysis, factor IC
3. XGBoost + SHAP — which features drive returns and in which direction
4. Walk-forward validation — honest out-of-sample performance
5. Live scoring — rank all current tickers by predicted return

---
## Cell 1 — Imports & Config

In [ ]:
import sys, warnings
from pathlib import Path

import duckdb
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.stats import spearmanr
from dotenv import load_dotenv
import os

warnings.filterwarnings('ignore')
plt.rcParams['figure.figsize'] = (13, 5)
plt.rcParams['axes.spines.top']   = False
plt.rcParams['axes.spines.right'] = False
sns.set_palette('husl')
pd.set_option('display.float_format', '{:.3f}'.format)
pd.set_option('display.max_columns', None)

ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT))
load_dotenv(ROOT / '.env', override=True)

from historic_fundamentals import get_pe_stats
from historic_fundamentals.db import HistoricFundamentalsDB

DB_PATH = os.getenv('HF_DB_PATH', str(ROOT / 'data' / 'historic_fundamentals.duckdb'))

# ── Return horizons: name -> months ahead ─────────────────────────────────────
RETURN_HORIZONS = {
    'ret_6m':  6,
    'ret_1y':  12,
    'ret_2y':  24,
    'ret_3y':  36,
    'ret_5y':  60,
}
RETURN_COLS   = list(RETURN_HORIZONS.keys())
PRIMARY_TARGET = 'ret_1y'   # 1-year horizon is the natural fit for fundamental investing

MIN_VALID      = 200         # minimum rows for a horizon to enter analysis
N_BINS         = 5           # quintiles for EDA
RANDOM_STATE   = 42
WINSOR_CLIP    = 3.0         # clip returns beyond ±300% (survivorship extremes)

print('Imports done.')
print('DB:', DB_PATH)

---
## Cell 2 — Load monthly_pe Data

In [ ]:
conn = duckdb.connect(DB_PATH, read_only=True)
raw  = conn.execute("""
    SELECT ticker, month_end_date, price,
           pe_ratio,    pe_rolling_5yr_median,
           pfcf_ratio,  pfcf_rolling_5yr_median, fcf_yield,
           ev_ebitda,   ev_ebitda_rolling_5yr_median,
           ps_ratio,    ps_rolling_5yr_median,
           roa,         roa_rolling_5yr_median,
           roe,         roe_rolling_5yr_median,
           roic,        roic_rolling_5yr_median,
           pbv,         pbv_rolling_5yr_median,
           ptbv,        ptbv_rolling_5yr_median,
           dividend_yield,
           goal_low, goal_high
    FROM monthly_pe
    WHERE price > 0
    ORDER BY ticker, month_end_date
""").df()
conn.close()

raw['month_end_date'] = pd.to_datetime(raw['month_end_date'])

print(f'Rows: {len(raw):,}  |  Tickers: {raw["ticker"].nunique():,}')
print(f'Date range: {raw["month_end_date"].min().date()} → {raw["month_end_date"].max().date()}')
raw.head(3)

---
## Cell 3 — Compute Forward Returns

For each (ticker, month T), the forward return at horizon N is:

    ret_N = price[T + N months] / price[T] - 1

We look for a price within ±45 days of the target date. Rows where the future price
is not yet available (too recent) will be NaN — handled in Cell 5.

In [ ]:
def compute_forward_returns(df, horizons, tolerance_days=45):
    """
    Add forward return columns using a vectorized merge approach.
    For each row, find the closest future price within tolerance_days of target date.
    """
    df = df.sort_values(['ticker', 'month_end_date']).reset_index(drop=True).copy()

    for col, months in horizons.items():
        # Build a "target date" column
        df['_target'] = df['month_end_date'] + pd.DateOffset(months=months)

        # Self-join on ticker, find all price rows within ±tolerance of target
        left  = df[['ticker', 'month_end_date', 'price', '_target']].copy()
        right = df[['ticker', 'month_end_date', 'price']].rename(
            columns={'month_end_date': 'future_date', 'price': 'future_price'})

        merged = left.merge(right, on='ticker', how='left')
        merged['day_diff'] = (merged['future_date'] - merged['_target']).abs().dt.days
        merged = merged[merged['day_diff'] <= tolerance_days]

        # Keep closest future date per (ticker, month_end_date)
        merged = (merged
                  .sort_values('day_diff')
                  .groupby(['ticker', 'month_end_date'], sort=False)
                  .first()
                  .reset_index()[['ticker', 'month_end_date', 'future_price']])

        df = df.merge(merged, on=['ticker', 'month_end_date'], how='left')
        df[col] = df['future_price'] / df['price'] - 1
        df = df.drop(columns=['future_price', '_target'])

        valid = df[col].notna().sum()
        print(f'  {col}: {valid:,} valid  '
              f'mean={df.loc[df[col].notna(), col].mean()*100:.1f}%  '
              f'median={df.loc[df[col].notna(), col].median()*100:.1f}%')

    return df

print('Computing forward returns...')
df = compute_forward_returns(raw, RETURN_HORIZONS)

# Winsorise: clip extreme returns (bankrupt / delisted / splits)
for col in RETURN_COLS:
    df[col] = df[col].clip(-WINSOR_CLIP, WINSOR_CLIP)

print(f'\nDone. Total rows: {len(df):,}')

---
## Cell 4 — Feature Engineering

Three feature groups:
- **Valuation level** — absolute multiples (PE, P/FCF, EV/EBITDA, P/S, P/BV, P/TBV, FCF yield)
- **Mean-reversion signal** — current multiple ÷ own 5yr median (>1 = premium, <1 = discount)
- **Quality / profitability** — ROA, ROE, ROIC
- **Analyst signal** — discount to SBB (goal_low), upside to target (goal_high) — NaN for tickers without GP data

In [ ]:
def safe_ratio(a, b, clip=10.0):
    """a / b, clipped to avoid extreme values from near-zero denominators."""
    return (a / b.replace(0, np.nan)).clip(-clip, clip)

# ── Mean-reversion premiums ───────────────────────────────────────────────────
df['pe_premium']   = safe_ratio(df['pe_ratio'],   df['pe_rolling_5yr_median'])
df['pfcf_premium'] = safe_ratio(df['pfcf_ratio'], df['pfcf_rolling_5yr_median'])
df['ev_premium']   = safe_ratio(df['ev_ebitda'],  df['ev_ebitda_rolling_5yr_median'])
df['ps_premium']   = safe_ratio(df['ps_ratio'],   df['ps_rolling_5yr_median'])
df['roa_premium']  = safe_ratio(df['roa'],        df['roa_rolling_5yr_median'])
df['roe_premium']  = safe_ratio(df['roe'],        df['roe_rolling_5yr_median'])
df['roic_premium'] = safe_ratio(df['roic'],       df['roic_rolling_5yr_median'])
df['pbv_premium']  = safe_ratio(df['pbv'],        df['pbv_rolling_5yr_median'])
df['ptbv_premium'] = safe_ratio(df['ptbv'],       df['ptbv_rolling_5yr_median'])

# NOTE: goal_discount / goal_upside are excluded — pe_lt_median (all-time) is used
# in their computation, which introduces look-ahead bias for historical training rows.

FEATURE_COLS = [
    # Valuation level
    'pe_ratio', 'pfcf_ratio', 'ev_ebitda', 'ps_ratio', 'pbv', 'ptbv',
    'fcf_yield', 'dividend_yield',
    # Mean-reversion signals (5yr rolling median — no look-ahead)
    'pe_premium', 'pfcf_premium', 'ev_premium', 'ps_premium',
    'pbv_premium', 'ptbv_premium',
    # Quality / profitability
    'roa', 'roe', 'roic',
    'roa_premium', 'roe_premium', 'roic_premium',
    # Historical medians (fair-value anchors, 5yr rolling)
    'pe_rolling_5yr_median', 'pfcf_rolling_5yr_median', 'ev_ebitda_rolling_5yr_median',
    'ps_rolling_5yr_median', 'roa_rolling_5yr_median', 'roe_rolling_5yr_median',
    'roic_rolling_5yr_median',
]

print(f'Features: {len(FEATURE_COLS)}')
print()
print('Feature coverage (fraction non-null, full dataset):'  )
coverage = df[FEATURE_COLS].notna().mean().sort_values(ascending=False)
for feat, pct in coverage.items():
    print(f'  {feat:35}: {pct:.0%}')

---
## Cell 5 — Return Target Masking

NaN forward returns mean the future price is not yet available (too recent to have N-month data).
We build a boolean mask per horizon and derive `EDA_TARGETS` automatically.

In [ ]:
for col in RETURN_COLS:
    df[col + '_valid'] = df[col].notna()

def get_valid(data, return_col):
    return data[data[return_col + '_valid']].copy()

valid_summary = pd.DataFrame({
    'valid_rows': [df[c + '_valid'].sum() for c in RETURN_COLS],
    'valid_pct':  [df[c + '_valid'].mean() * 100 for c in RETURN_COLS],
    'mean_ret':   [df.loc[df[c + '_valid'], c].mean() * 100 for c in RETURN_COLS],
    'median_ret': [df.loc[df[c + '_valid'], c].median() * 100 for c in RETURN_COLS],
}, index=RETURN_COLS).round(1)

print('Forward return availability and summary:')
print(valid_summary.to_string())

EDA_TARGETS      = [c for c in RETURN_COLS if df[c + '_valid'].sum() >= MIN_VALID]
ML_TARGETS       = EDA_TARGETS
PRIMARY_ML_TARGET = PRIMARY_TARGET if PRIMARY_TARGET in EDA_TARGETS else EDA_TARGETS[0]

print(f'\nEDA / ML targets: {EDA_TARGETS}')
print(f'Primary ML target: {PRIMARY_ML_TARGET}')

---
## Cell 6 — EDA: Feature Correlation Heatmap

Pearson correlation of each feature with each return horizon.
Green = positive predictor of returns, red = negative.

In [ ]:
df_valid_primary = get_valid(df, PRIMARY_ML_TARGET)

corr_targets = [c for c in RETURN_COLS if df[c + '_valid'].sum() >= MIN_VALID]
corr_matrix  = (
    df_valid_primary[FEATURE_COLS + corr_targets]
    .corr()[corr_targets]
    .loc[FEATURE_COLS]
)

fig, ax = plt.subplots(figsize=(len(corr_targets) * 2 + 3, len(FEATURE_COLS) * 0.45 + 2))
sns.heatmap(
    corr_matrix, annot=True, fmt='.2f', center=0,
    cmap='RdYlGn', linewidths=0.4, ax=ax,
    vmin=-0.15, vmax=0.15
)
ax.set_title('Feature vs Forward Return Correlation  (green=positive, red=negative)',
             fontsize=12, pad=10)
plt.tight_layout()
plt.show()

print(f'Top features by |correlation| with {PRIMARY_ML_TARGET}:')
top = corr_matrix[PRIMARY_ML_TARGET].abs().sort_values(ascending=False)
print(top.round(3).to_string())

---
## Cell 7 — EDA: Factor IC (Information Coefficient)

The Information Coefficient (IC) measures how well a feature ranks stocks in order of their
subsequent returns. Computed as the **monthly cross-sectional Spearman rank correlation**
between each feature and the forward return.

- IC > 0 means higher feature value → higher return
- ICIR = mean(IC) / std(IC) — analogous to Sharpe ratio for a factor
- Rule of thumb: |ICIR| > 0.5 is practically significant

In [ ]:
def compute_ic(data, features, targets, min_stocks=20):
    """Monthly cross-sectional Spearman IC for each (feature, target) pair."""
    records = []
    for target in targets:
        sub = get_valid(data, target)
        for month, mdf in sub.groupby('month_end_date'):
            if len(mdf) < min_stocks:
                continue
            for feat in features:
                valid = mdf[[feat, target]].dropna()
                if len(valid) < min_stocks:
                    continue
                ic, _ = spearmanr(valid[feat], valid[target])
                records.append({'month': month, 'feature': feat, 'target': target, 'ic': ic})
    return pd.DataFrame(records)

print('Computing monthly IC (this may take ~1-2 minutes)...')
ic_df = compute_ic(df, FEATURE_COLS, EDA_TARGETS)

ic_summary = (
    ic_df.groupby(['feature', 'target'])['ic']
    .agg(mean_ic='mean', std_ic='std', months='count')
    .reset_index()
)
ic_summary['icir'] = ic_summary['mean_ic'] / ic_summary['std_ic']

# Pivot to show mean IC per (feature, target)
ic_pivot = ic_summary.pivot(index='feature', columns='target', values='mean_ic')
ic_pivot = ic_pivot.reindex(columns=EDA_TARGETS)
ic_pivot['abs_mean'] = ic_pivot.abs().mean(axis=1)
ic_pivot = ic_pivot.sort_values('abs_mean', ascending=False).drop(columns='abs_mean')

print('\nMean monthly IC by feature and horizon (Spearman):')
print(ic_pivot.round(3).to_string())

# Heatmap
fig, ax = plt.subplots(figsize=(len(EDA_TARGETS) * 2 + 3, len(FEATURE_COLS) * 0.45 + 2))
sns.heatmap(
    ic_pivot, annot=True, fmt='.3f', center=0,
    cmap='RdYlGn', linewidths=0.4, ax=ax,
    vmin=-0.06, vmax=0.06
)
ax.set_title('Mean Monthly IC (Spearman rank correlation per month, averaged)', fontsize=12, pad=10)
plt.tight_layout()
plt.show()

# ICIR for primary target
icir_primary = (
    ic_summary[ic_summary['target'] == PRIMARY_ML_TARGET]
    .set_index('feature')['icir']
    .sort_values(key=abs, ascending=False)
)
print(f'\nICIR for {PRIMARY_ML_TARGET} (|ICIR|>0.5 = practically significant):')
print(icir_primary.round(3).to_string())

---
## Cell 8 — EDA: Quintile Return Analysis

For each feature, split stocks into 5 equal-frequency buckets (quintiles) and compare
average forward returns. A monotonic pattern (Q1 < Q2 < Q3 < Q4 < Q5 or vice versa)
signals a robust factor.

In [ ]:
KEY_FEATURES = [
    'pe_premium', 'pfcf_premium', 'ev_premium', 'ps_premium',
    'roic', 'roe', 'roa',
    'fcf_yield', 'dividend_yield',
    'goal_discount', 'goal_upside',
]

def quintile_returns(data, feature, targets, n_bins=5):
    """Mean return by quintile for one feature across all targets."""
    rows = {}
    for target in targets:
        sub = get_valid(data, target)[[feature, target]].dropna()
        if len(sub) < 100:
            continue
        sub['bucket'] = pd.qcut(sub[feature], q=n_bins, labels=False, duplicates='drop')
        rows[target] = sub.groupby('bucket')[target].mean() * 100
    return pd.DataFrame(rows) if rows else pd.DataFrame()

n_feats   = len(KEY_FEATURES)
n_targets = len(EDA_TARGETS)
fig, axes = plt.subplots(n_feats, 1, figsize=(13, 4.5 * n_feats))
if n_feats == 1:
    axes = [axes]

colors = plt.cm.tab10.colors

for ax, feat in zip(axes, KEY_FEATURES):
    qt = quintile_returns(df, feat, EDA_TARGETS, N_BINS)
    if qt.empty:
        ax.set_title(f'{feat}  (insufficient data)')
        continue
    x = np.arange(len(qt))
    width = 0.8 / max(len(EDA_TARGETS), 1)
    for i, target in enumerate(EDA_TARGETS):
        if target not in qt.columns:
            continue
        ax.bar(x + i * width, qt[target], width=width * 0.9,
               label=target, color=colors[i % len(colors)], alpha=0.8)
    ax.axhline(0, color='black', linewidth=0.6)
    ax.set_xticks(x + width * (n_targets - 1) / 2)
    ax.set_xticklabels([f'Q{i+1}' for i in range(len(qt))], fontsize=9)
    ax.set_ylabel('Mean return (%)')
    ax.set_title(f'{feat}  —  mean return by quintile  (Q1=lowest, Q5=highest)')
    ax.legend(fontsize=8, loc='upper left')

plt.tight_layout()
plt.show()

---
## Cell 9 — ML: XGBoost Feature Importance

One XGBoost regressor per return horizon. SHAP values reveal which features drive
predictions and in which direction.

In [ ]:
from xgboost import XGBRegressor
import shap

MIN_ROWS_TO_TRAIN = 500

def get_xgb_params(n_rows):
    """Scale regularisation with dataset size."""
    if n_rows < 2_000:
        return dict(n_estimators=100, max_depth=3, learning_rate=0.05,
                    subsample=0.8, colsample_bytree=0.8, min_child_weight=20)
    return dict(n_estimators=400, max_depth=4, learning_rate=0.02,
                subsample=0.8, colsample_bytree=0.8, min_child_weight=20)

models     = {}   # {target: fitted XGBRegressor}
ml_datasets = {}  # {target: (X, y)}

print('Training XGBoost models...')
print(f'{"Horizon":<12} {"Rows":>8}  {"Features":>10}  Status')
print('-' * 50)

for target in ML_TARGETS:
    sub = get_valid(df, target)[FEATURE_COLS + [target, 'month_end_date', 'ticker']]
    sub = sub.dropna(subset=[target])

    # Fill feature NaNs with column median (no leakage — done per-target, pre-split)
    X = sub[FEATURE_COLS].fillna(sub[FEATURE_COLS].median())
    y = sub[target]

    if len(X) < MIN_ROWS_TO_TRAIN:
        print(f'{target:<12} {len(X):>8}  skipped (< {MIN_ROWS_TO_TRAIN} rows)')
        continue

    params = get_xgb_params(len(X))
    model  = XGBRegressor(**params, random_state=RANDOM_STATE, verbosity=0)
    model.fit(X, y)

    models[target]      = model
    ml_datasets[target] = (X.values, y.values)

    # In-sample R²
    r2 = 1 - np.sum((y - model.predict(X))**2) / np.sum((y - y.mean())**2)
    print(f'{target:<12} {len(X):>8}  {len(FEATURE_COLS):>10}  R²={r2:.3f}  (in-sample)')

PRIMARY_ML_TARGET = next((t for t in ML_TARGETS if t in models), None)
print(f'\nPrimary target for walk-forward: {PRIMARY_ML_TARGET}')

---
## Cell 10 — SHAP Feature Importance Plots

Bar chart: which features matter most (magnitude).  
Beeswarm: direction of effect (high feature value → positive or negative return?).

In [ ]:
if not models:
    print('No models trained. Run Cell 9 first.')
else:
    trained = list(models.keys())
    fig, axes = plt.subplots(len(trained), 2, figsize=(16, 5.5 * len(trained)))
    if len(trained) == 1:
        axes = [axes]

    shap_importance = {}

    for row_idx, target in enumerate(trained):
        model_t = models[target]
        X_t, y_t = ml_datasets[target]

        # Use a sample for SHAP speed on large datasets
        sample_size = min(5_000, len(X_t))
        idx = np.random.RandomState(RANDOM_STATE).choice(len(X_t), sample_size, replace=False)
        X_sample = X_t[idx]

        explainer   = shap.TreeExplainer(model_t)
        shap_values = explainer.shap_values(X_sample)

        shap_importance[target] = pd.Series(
            np.abs(shap_values).mean(axis=0),
            index=FEATURE_COLS
        ).sort_values(ascending=False)

        ax_bar = axes[row_idx][0]
        plt.sca(ax_bar)
        shap.summary_plot(shap_values, X_sample, feature_names=FEATURE_COLS,
                          plot_type='bar', show=False, max_display=15, color='steelblue')
        ax_bar.set_title(f'{target}  |  Feature importance (mean |SHAP|)',
                         fontsize=10, fontweight='bold')

        ax_bee = axes[row_idx][1]
        plt.sca(ax_bee)
        shap.summary_plot(shap_values, X_sample, feature_names=FEATURE_COLS,
                          show=False, max_display=15)
        ax_bee.set_title(f'{target}  |  Direction of effect (beeswarm)',
                         fontsize=10, fontweight='bold')

    plt.tight_layout()
    plt.show()

    # Consolidated importance table
    imp_df = pd.DataFrame(shap_importance).round(5)
    imp_df['mean'] = imp_df.mean(axis=1)
    imp_df = imp_df.sort_values('mean', ascending=False)
    print('\nConsolidated SHAP importance across all horizons (top 15):')
    print(imp_df.head(15).to_string())

---
## Cell 11 — Walk-Forward Validation

Proper time-series validation: train on all months up to T, predict month T+1.
This avoids lookahead bias. We compare:
- **ML top-decile**: stocks ranked highest by model prediction
- **Universe equal-weight**: all stocks with valid data (benchmark)

In [ ]:
from sklearn.metrics import r2_score as sk_r2

WF_TARGET       = PRIMARY_ML_TARGET
MIN_TRAIN_MONTHS = 60  # 5 years of history before first prediction
TOP_DECILE_PCT   = 0.20  # top 20% of ranked stocks

wf_cols  = FEATURE_COLS + [WF_TARGET, 'month_end_date', 'ticker']
ml_dated = get_valid(df, WF_TARGET)[wf_cols].copy()
ml_dated = ml_dated.dropna(subset=[WF_TARGET])

dates_sorted = sorted(ml_dated['month_end_date'].unique())
n_test = len(dates_sorted) - MIN_TRAIN_MONTHS

print(f'Walk-forward target : {WF_TARGET}')
print(f'Total months        : {len(dates_sorted)}')
print(f'Training window     : first {MIN_TRAIN_MONTHS} months')
print(f'Test months         : {n_test}')

if n_test < 12:
    print(f'WARNING: only {n_test} test months — results not meaningful.')
    wf_df = pd.DataFrame()
else:
    wf_results = []

    for i in range(MIN_TRAIN_MONTHS, len(dates_sorted)):
        train_dates = dates_sorted[:i]
        test_date   = dates_sorted[i]

        train = ml_dated[ml_dated['month_end_date'].isin(train_dates)]
        test  = ml_dated[ml_dated['month_end_date'] == test_date].copy()

        if len(test) < 5:
            continue

        X_tr = train[FEATURE_COLS].fillna(train[FEATURE_COLS].median())
        y_tr = train[WF_TARGET]
        X_te = test[FEATURE_COLS].fillna(train[FEATURE_COLS].median())  # train medians only
        y_te = test[WF_TARGET]

        params = get_xgb_params(len(X_tr))
        wf_model = XGBRegressor(**params, random_state=RANDOM_STATE, verbosity=0)
        wf_model.fit(X_tr, y_tr)

        preds = wf_model.predict(X_te)
        test['pred'] = preds

        n_top = max(1, int(len(test) * TOP_DECILE_PCT))
        top_ret  = test.nlargest(n_top, 'pred')[WF_TARGET].mean()
        bench_ret = y_te.mean()

        ic, _ = spearmanr(preds, y_te)

        wf_results.append({
            'month':       test_date,
            'train_months': i,
            'n_stocks':    len(test),
            'top_ret':     top_ret,
            'bench_ret':   bench_ret,
            'excess_ret':  top_ret - bench_ret,
            'monthly_ic':  ic,
        })

    wf_df = pd.DataFrame(wf_results)

    print(f'\nWalk-forward results ({len(wf_df)} months tested):')
    print(f'  Mean top-{TOP_DECILE_PCT:.0%} return  : {wf_df["top_ret"].mean()*100:+.2f}%')
    print(f'  Mean benchmark return : {wf_df["bench_ret"].mean()*100:+.2f}%')
    print(f'  Mean excess return    : {wf_df["excess_ret"].mean()*100:+.2f}%')
    print(f'  Win rate vs benchmark : {(wf_df["excess_ret"]>0).mean():.0%}')
    print(f'  Mean monthly IC       : {wf_df["monthly_ic"].mean():.4f}')
    icir = wf_df['monthly_ic'].mean() / wf_df['monthly_ic'].std()
    print(f'  ICIR                  : {icir:.3f}')

---
## Cell 12 — Walk-Forward: Cumulative Return Chart

In [ ]:
if 'wf_df' not in dir() or wf_df.empty:
    print('No walk-forward results. Run Cell 11 first.')
else:
    cum_top   = (1 + wf_df.set_index('month')['top_ret']).cumprod() - 1
    cum_bench = (1 + wf_df.set_index('month')['bench_ret']).cumprod() - 1

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    # Cumulative return
    ax1.plot(cum_bench.index, cum_bench * 100, 'k--', lw=1.5, label='Universe EW')
    ax1.plot(cum_top.index,   cum_top   * 100, color='steelblue', lw=2,
             label=f'ML top {TOP_DECILE_PCT:.0%}')
    ax1.set_ylabel('Cumulative return (%)')
    ax1.set_title(f'Walk-Forward Cumulative Return  ({WF_TARGET})')
    ax1.legend()

    # Monthly IC over time
    ax2.bar(wf_df['month'], wf_df['monthly_ic'],
            color=np.where(wf_df['monthly_ic'] >= 0, 'steelblue', 'coral'),
            alpha=0.7)
    ax2.axhline(0, color='black', lw=0.8)
    ax2.axhline(wf_df['monthly_ic'].mean(), color='red', lw=1.2, linestyle='--',
                label=f'Mean IC={wf_df["monthly_ic"].mean():.3f}')
    ax2.set_ylabel('Monthly IC (Spearman)')
    ax2.set_title('Monthly Information Coefficient')
    ax2.legend()

    plt.tight_layout()
    plt.show()

---
## Cell 13 — Live Scoring: Rank All Current Tickers

Apply the trained model to today's fundamentals from `pe_stats`.
Output: ranked list of tickers by predicted forward return.

In [ ]:
if PRIMARY_ML_TARGET not in models:
    print(f'No model for {PRIMARY_ML_TARGET}. Run Cell 9 first.')
else:
    live = get_pe_stats()
    print(f'Tickers in pe_stats: {len(live):,}')

    def _col(name):
        return live[name] if name in live.columns else pd.Series(np.nan, index=live.index)

    fdf = pd.DataFrame(index=live.index)
    fdf['pe_ratio']                    = _col('current_pe')
    fdf['pfcf_ratio']                  = _col('current_pfcf')
    fdf['ev_ebitda']                   = _col('current_evebitda')
    fdf['ps_ratio']                    = _col('current_ps')
    fdf['pbv']                         = _col('current_pbv')
    fdf['ptbv']                        = _col('current_ptbv')
    fdf['fcf_yield']                   = _col('current_fcf_yield')
    fdf['dividend_yield']              = _col('dividend_yield')
    fdf['roa']                         = _col('current_roa')
    fdf['roe']                         = _col('current_roe')
    fdf['roic']                        = _col('current_roic')
    fdf['pe_rolling_5yr_median']       = _col('pe_rolling_5yr_median')
    fdf['pfcf_rolling_5yr_median']     = _col('pfcf_rolling_5yr_median')
    fdf['ev_ebitda_rolling_5yr_median']= _col('evebitda_rolling_5yr_median')
    fdf['ps_rolling_5yr_median']       = _col('ps_rolling_5yr_median')
    fdf['roa_rolling_5yr_median']      = _col('roa_rolling_5yr_median')
    fdf['roe_rolling_5yr_median']      = _col('roe_rolling_5yr_median')
    fdf['roic_rolling_5yr_median']     = _col('roic_rolling_5yr_median')
    fdf['pbv_rolling_5yr_median']      = _col('pbv_rolling_5yr_median')
    fdf['ptbv_rolling_5yr_median']     = _col('ptbv_rolling_5yr_median')

    fdf['pe_premium']   = safe_ratio(fdf['pe_ratio'],   fdf['pe_rolling_5yr_median'])
    fdf['pfcf_premium'] = safe_ratio(fdf['pfcf_ratio'], fdf['pfcf_rolling_5yr_median'])
    fdf['ev_premium']   = safe_ratio(fdf['ev_ebitda'],  fdf['ev_ebitda_rolling_5yr_median'])
    fdf['ps_premium']   = safe_ratio(fdf['ps_ratio'],   fdf['ps_rolling_5yr_median'])
    fdf['roa_premium']  = safe_ratio(fdf['roa'],        fdf['roa_rolling_5yr_median'])
    fdf['roe_premium']  = safe_ratio(fdf['roe'],        fdf['roe_rolling_5yr_median'])
    fdf['roic_premium'] = safe_ratio(fdf['roic'],       fdf['roic_rolling_5yr_median'])
    fdf['pbv_premium']  = safe_ratio(fdf['pbv'],        fdf['pbv_rolling_5yr_median'])
    fdf['ptbv_premium'] = safe_ratio(fdf['ptbv'],       fdf['ptbv_rolling_5yr_median'])

    train_medians = ml_dated[FEATURE_COLS].median()
    X_live = fdf[FEATURE_COLS].copy()
    for col in FEATURE_COLS:
        X_live[col] = X_live[col].fillna(train_medians.get(col, 0))

    live['pred_ret'] = models[PRIMARY_ML_TARGET].predict(X_live.values)
    live['ml_rank']  = live['pred_ret'].rank(ascending=False).astype(int)

    top50 = live.nsmallest(50, 'ml_rank')[[
        'ticker', 'ml_rank', 'current_price', 'pred_ret',
        'market_cap_b', 'forward_pe', 'current_roic', 'current_ps',
    ]].copy()
    top50['pred_ret_pct'] = (top50['pred_ret'] * 100).round(1)
    top50 = top50.drop(columns='pred_ret')

    print(f'\nTop 50 by predicted {PRIMARY_ML_TARGET} return:')
    print(top50.round(2).to_string(index=False))

---
## Cell 14 — Summary

In [ ]:
print('=' * 65)
print('  FUNDAMENTALS ALPHA — SUMMARY')
print('=' * 65)
print(f'  Data:           {len(df):,} monthly observations, {df["ticker"].nunique():,} tickers')
print(f'  Date range:     {df["month_end_date"].min().date()} → {df["month_end_date"].max().date()}')
print(f'  Features:       {len(FEATURE_COLS)}')
print(f'  Primary target: {PRIMARY_ML_TARGET}')
print()

if models:
    print('── MODELS TRAINED ───────────────────────────────────────────────')
    for target in ML_TARGETS:
        if target in models:
            X_t, y_t = ml_datasets[target]
            preds = models[target].predict(X_t)
            r2 = 1 - np.sum((y_t - preds)**2) / np.sum((y_t - y_t.mean())**2)
            print(f'  {target:<10} rows={len(X_t):>7,}  in-sample R²={r2:.3f}')

if 'wf_df' in dir() and not wf_df.empty:
    print()
    print('── WALK-FORWARD RESULTS ─────────────────────────────────────────')
    icir = wf_df['monthly_ic'].mean() / wf_df['monthly_ic'].std()
    print(f'  Target         : {WF_TARGET}')
    print(f'  Test months    : {len(wf_df)}')
    print(f'  Excess return  : {wf_df["excess_ret"].mean()*100:+.2f}% / month vs universe')
    print(f'  Win rate       : {(wf_df["excess_ret"]>0).mean():.0%}')
    print(f'  Mean IC        : {wf_df["monthly_ic"].mean():.4f}')
    print(f'  ICIR           : {icir:.3f}')

print()
print('To interpret results:')
print('  - Negative IC for PE/PS/EV premiums = cheaper stocks outperform (value factor)')
print('  - Positive IC for ROIC/ROE = higher quality outperforms (quality factor)')
print('  - ICIR > 0.5 = practically significant factor')
print('  - ICIR > 1.0 = strong, consistent factor worth trading')